# Configurable support intent and triage
**Practical notebook 02 · Unvibecode**

“Send my invoice, cancel my subscription, and refund the extra charge. This is my third message.”

A single intent label loses part of that request. This notebook separates **business intent, clarity, scope, and handling**, then uses those decisions in a small support application.

**Part 1:** Editable policy, synthetic training data, a local SetFit classifier, and deterministic checks.  
**Part 2:** A conversation with mock support handlers, failure-path tests, and measured classification/routing performance.

The application profile is subscription support in English. The model is `all-MiniLM-L6-v2` with SetFit's multilabel logistic-regression head. Training is local; no generative LLM, API key, or GPU is required. The examples are a demonstration dataset, not a pretrained production support model.

## Setup
Use Python 3.12 in a fresh environment. From a terminal:

```bash
python -m venv .venv
```

Activate with `source .venv/bin/activate` on macOS/Linux, or `.venv\Scripts\Activate.ps1` in Windows PowerShell. Then run:

```bash
python -m pip install jupyterlab
python -m jupyterlab
```

Open this notebook. Set `INSTALL_DEPENDENCIES=True` below and run only that cell. Restart the kernel, change it back to `False`, and choose **Run All**. The first model download needs internet. The training cell runs 40 contrastive training steps and fits the classification heads; runtime depends on your CPU.

The CPU PyTorch wheel is selected explicitly. These are tested dependency versions, not a claim that they are the newest releases. Training writes temporary trainer output to `intent_training_output`; the notebook does not send messages to a live support service.

In [ ]:
import subprocess
import sys

INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch==2.6.0",
                           "--index-url", "https://download.pytorch.org/whl/cpu"])
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "setfit==1.1.3", "sentence-transformers==3.4.1",
                           "transformers==4.51.3", "datasets==3.5.0", "scikit-learn==1.6.1"])
    print("Restart the kernel, set INSTALL_DEPENDENCIES=False, then Run All.")

# Part 1 — Preconfigurable classification and policy

| Dimension | Output | What controls it |
|---|---|---|
| Intent | Invoice, duplicate charge, payment failure, refund, cancellation, account access; multiple labels allowed | SetFit trained on labeled examples |
| Clarity | Clear, missing context, conflicting, uncertain; separate multiple-request flag | Required fields, conversation state, narrow conflict rules |
| Scope | Supported, partial, unsupported, uncertain | Predicted intents plus enabled application capabilities |
| Handling | Routine/time-sensitive/immediate review, frustration signal, next action, destination | Learned urgency/frustration and explicit routing policy |

Billing and payments are distinct destinations. Invoice, duplicate-charge, and refund requests route to billing; payment failures route to payments. A mixed request can reach both.

`unsupported` is trained on example exclusions; it does not exhaust every possible topic. A classifier can confidently misclassify unfamiliar requests. No accepted intent becomes `uncertain`, not automatically `unsupported`. The evaluation below exposes some of those mistakes.

In [ ]:
import os
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

import numpy as np
INTENTS = ("invoice", "duplicate_charge", "payment_failure", "refund", "cancel", "account_access")
SIGNALS = ("unsupported", "urgent", "frustrated")
LABELS = INTENTS + SIGNALS

### 1.1 Policy and structured results
Edit required fields for the next operation, not for every message about a topic. In this narrow profile, `refund` means a request to investigate a particular purchase and requires a reference. General refund-policy Q&A would need a separate intent and training examples; it should not require an order reference.

The application supplies validated conversation context separately. Only a narrow slot-only reply can resume pending intents. An unrelated new request is classified on its own. Context is not proof of identity or resource ownership.

The two critical patterns are illustrative review triggers, not a comprehensive security detector. Keyword matches can be quoted or negated; a match routes to review and never grants permission to execute an action.

In [ ]:
from dataclasses import dataclass, field, replace, asdict
from types import MappingProxyType
from typing import Mapping
from time import perf_counter
from collections import Counter
import hashlib
import json
import re


@dataclass(frozen=True)
class RouterConfig:
    supported_intents: tuple[str, ...] = INTENTS
    thresholds: Mapping[str, float] = field(default_factory=dict)
    max_chars: int = 2000
    max_clarifications: int = 2
    frustration_handoff: bool = True
    routes: Mapping[str, str] = field(default_factory=lambda: {
        "invoice": "billing", "duplicate_charge": "billing", "refund": "billing",
        "payment_failure": "payments", "cancel": "subscriptions", "account_access": "account_support",
    })
    required_fields: Mapping[str, tuple[str, ...]] = field(default_factory=lambda: {
        "refund": ("order_reference",), "duplicate_charge": ("order_reference",),
        "cancel": ("target",),
    })
    critical_patterns: tuple[str, ...] = (
        r"\b(?:someone is using my account|my account (?:has been |is )hacked)\b",
        r"\bcheckout is down for (?:everyone|all customers)\b",
    )

    def __post_init__(self):
        if not self.supported_intents or not set(self.supported_intents) <= set(INTENTS):
            raise ValueError("Choose supported intents from the trained taxonomy.")
        if type(self.max_chars) is not int or self.max_chars < 1:
            raise ValueError("max_chars must be positive.")
        if type(self.max_clarifications) is not int or self.max_clarifications < 1:
            raise ValueError("max_clarifications must be positive.")
        if type(self.frustration_handoff) is not bool:
            raise ValueError("frustration_handoff must be boolean.")
        if set(self.thresholds) - set(LABELS):
            raise ValueError("Unknown threshold label.")
        if any(type(v) not in (int, float) or not np.isfinite(v) or not 0 < v < 1
               for v in self.thresholds.values()):
            raise ValueError("Thresholds must be between zero and one.")
        if not set(self.supported_intents) <= set(self.routes):
            raise ValueError("Every supported intent needs a route.")
        if set(self.required_fields) - set(INTENTS):
            raise ValueError("Unknown required-field intent.")
        if any(set(fields) - {"target", "order_reference"} for fields in self.required_fields.values()):
            raise ValueError("Only target and order_reference fields are implemented.")
        for pattern in self.critical_patterns:
            re.compile(pattern)
        object.__setattr__(self, "supported_intents", tuple(self.supported_intents))
        object.__setattr__(self, "critical_patterns", tuple(self.critical_patterns))
        for key in ("thresholds", "routes", "required_fields"):
            value = dict(getattr(self, key))
            if key == "required_fields":
                value = {name: tuple(fields) for name, fields in value.items()}
            object.__setattr__(self, key, MappingProxyType(value))

    def document(self):
        return {"supported_intents": list(self.supported_intents),
                "thresholds": dict(self.thresholds), "routes": dict(self.routes),
                "required_fields": dict(self.required_fields), "max_chars": self.max_chars,
                "max_clarifications": self.max_clarifications,
                "frustration_handoff": self.frustration_handoff,
                "critical_patterns": list(self.critical_patterns)}

    @property
    def policy_id(self):
        return hashlib.sha256(json.dumps(self.document(), sort_keys=True).encode()).hexdigest()[:16]


@dataclass(frozen=True)
class ConversationContext:
    target: str | None = None
    order_reference: str | None = None
    pending_intents: tuple[str, ...] = ()
    pending_unsupported: bool = False
    clarification_count: int = 0

    def __post_init__(self):
        if self.target not in (None, "subscription", "order"):
            raise ValueError("Unknown context target.")
        if not set(self.pending_intents) <= set(INTENTS):
            raise ValueError("Unknown pending intent.")
        object.__setattr__(self, "pending_intents", tuple(self.pending_intents))
        if self.order_reference is not None and not re.fullmatch(r"ORDER-\d{3,8}", self.order_reference):
            raise ValueError("Expected a synthetic ORDER reference.")
        if type(self.clarification_count) is not int or self.clarification_count < 0:
            raise ValueError("Invalid clarification count.")


@dataclass(frozen=True)
class Decision:
    intents: tuple[str, ...]
    scope: str
    clarity: str
    multiple_requests: bool
    missing_fields: tuple[str, ...]
    urgency: str
    frustrated: bool
    action: str
    routes: tuple[str, ...]
    reason: str
    policy_id: str
    latency_ms: float

    def audit(self):
        return asdict(self)


class SetFitScorer:
    def __init__(self, model):
        self.model = model

    def scores(self, text):
        tokens = self.model.model_body.tokenizer(text, truncation=False)["input_ids"]
        if len(tokens) > self.model.model_body.max_seq_length:
            raise ValueError("Input exceeds the encoder token limit.")
        values = np.asarray(self.model.predict_proba([text], as_numpy=True))[0]
        if values.shape != (len(LABELS),) or not np.isfinite(values).all():
            raise ValueError("Invalid model scores.")
        return dict(zip(LABELS, map(float, values)))


def extract_fields(text, context):
    reference = re.search(r"\bORDER-\d{3,8}\b", text, re.I)
    target = context.target
    if re.search(r"\b(subscription|plan|membership|renewal)\b", text, re.I):
        target = "subscription"
    elif re.search(r"\b(?:cancel|stop) (?:my |the )?order\b", text, re.I):
        target = "order"
    return {"target": target,
            "order_reference": reference.group().upper() if reference else context.order_reference}


class SupportRouter:
    def __init__(self, scorer, config):
        self.scorer = scorer
        self.config = config

    def classify(self, text, context=None):
        started = perf_counter()
        context = ConversationContext() if context is None else context
        cfg = self.config

        def result(intents=(), scope="uncertain", clarity="uncertain", multiple=False,
                   missing=(), urgency="routine", frustrated=False, action="handoff",
                   routes=(), reason="CLASSIFIER_UNAVAILABLE"):
            return Decision(tuple(sorted(intents)), scope, clarity, multiple, tuple(sorted(missing)),
                            urgency, frustrated, action, tuple(sorted(set(routes))), reason,
                            cfg.policy_id, round((perf_counter() - started) * 1000, 3))

        if not isinstance(text, str) or not text.strip() or len(text) > cfg.max_chars:
            return result(reason="INVALID_OR_OVERSIZED_INPUT")
        if not isinstance(context, ConversationContext):
            return result(reason="INVALID_CONTEXT")
        try:
            scores = self.scorer.scores(text)
            if set(scores) != set(LABELS) or any(not np.isfinite(v) or not 0 <= v <= 1 for v in scores.values()):
                raise ValueError("Invalid scores.")
        except Exception:
            return result()
        selected = {label for label in LABELS if scores[label] >= cfg.thresholds.get(label, 0.5)}
        fields = extract_fields(text, context)
        slot_reply = bool(re.fullmatch(r"\s*(?:ORDER-\d{3,8}|(?:my |the )?(?:subscription|order))[.!]?\s*", text, re.I))
        if context.pending_intents and slot_reply:
            selected = set(context.pending_intents)
            if context.pending_unsupported:
                selected.add("unsupported")
            if re.fullmatch(r"\s*(?:my |the )?order[.!]?\s*", text, re.I):
                fields["target"] = "order"
        vague_cancel = bool(re.fullmatch(r"\s*(?:please |can you )?cancel (?:it|this)(?: for me)?[.!?]?\s*", text, re.I))
        if vague_cancel:
            selected.add("cancel")
        intents = selected & set(INTENTS)
        supported = intents & set(cfg.supported_intents)
        outside = bool("unsupported" in selected or intents - set(cfg.supported_intents))
        scope = "partial" if supported and outside else "supported" if supported else "unsupported" if outside else "uncertain"
        multiple = len(intents) + int("unsupported" in selected) > 1
        frustrated = "frustrated" in selected
        critical = any(re.search(pattern, text, re.I) for pattern in cfg.critical_patterns)
        urgency = "immediate_review" if critical else "time_sensitive" if "urgent" in selected else "routine"
        missing = {f for intent in supported for f in cfg.required_fields.get(intent, ()) if not fields.get(f)}
        conflict = bool("cancel" in intents and re.search(r"\b(?:keep|continue) (?:auto.?renewing|renewing|my subscription active)\b", text, re.I))
        clarity = "conflicting" if conflict else "missing_context" if missing else "uncertain" if scope == "uncertain" else "clear"
        routes = [cfg.routes[intent] for intent in supported]
        if critical:
            action, reason, routes = "handoff", "CRITICAL_REVIEW", ["incident_review"]
        elif frustrated and cfg.frustration_handoff:
            action, reason = "handoff", "FRUSTRATION_REVIEW"
        elif scope == "unsupported":
            action, reason = "redirect", "OUTSIDE_CONFIGURED_SCOPE"
        elif clarity != "clear":
            action = "handoff" if context.clarification_count >= cfg.max_clarifications else "clarify"
            reason = "CLARIFICATION_LIMIT" if action == "handoff" else clarity.upper()
        elif scope == "partial":
            action, reason = "route_supported", "PARTIAL_SCOPE"
        else:
            action, reason = "route", "SUPPORTED_REQUEST"
        return result(intents, scope, clarity, multiple, missing, urgency, frustrated, action, routes, reason)

In [ ]:
SUPPORTED_INTENTS = INTENTS
THRESHOLD_OVERRIDES = {}
MAX_CLARIFICATIONS = 2
FRUSTRATION_HANDOFF = True

base_config = RouterConfig(
    supported_intents=SUPPORTED_INTENTS,
    thresholds=THRESHOLD_OVERRIDES,
    max_clarifications=MAX_CLARIFICATIONS,
    frustration_handoff=FRUSTRATION_HANDOFF,
)
print(json.dumps(base_config.document(), indent=2))

### 1.2 Training, validation, and held-out examples
The synthetic data includes multiple intents, negation, mixed scope, urgency, and frustration. All examples are embedded in this notebook.

Training updates the encoder and classification heads. Validation selects one threshold per label from a fixed grid. The test examples are used only for reporting. The validation set is deliberately small: its thresholds can overfit and are not calibrated confidence probabilities. For deployment, replace it with representative labeled data and choose thresholds using the cost of each error.

Disabling an existing intent changes scope configuration. Adding an intent requires adding labeled data, retraining, and reevaluation. Exact-text overlap is checked below; production splits must also separate related conversations, customers, and paraphrase families.

In [ ]:
TRAIN_GROUPS = {
    "invoice": [
        "Please email my subscription invoice.", "I need a receipt for last month's bill.",
        "Where can I download a tax invoice?", "Send me the invoice for my purchase.",
        "Can I get a copy of my billing statement?", "I need proof of payment for expenses.",
        "Please show the invoice for my account.", "How do I obtain my latest receipt?",
        "I am not requesting a refund; I only need an invoice.",
        "Do not cancel anything. Just send my invoice.",
    ],
    "duplicate_charge": [
        "You charged my card twice.", "There are two charges for one subscription.",
        "The same payment appears twice on my statement.", "I was double billed this month.",
        "Why did you debit my account two times?", "I see duplicate charges for one order.",
        "One purchase resulted in two card deductions.", "Investigate a repeated subscription charge.",
    ],
    "payment_failure": [
        "My card was declined at checkout.", "I cannot complete my payment.",
        "Checkout shows a payment error.", "The transaction failed when I tried to pay.",
        "My subscription payment keeps failing.", "The payment page rejects my card.",
        "I tried to pay but the charge did not go through.", "Help me fix a declined transaction.",
    ],
    "refund": [
        "Please refund my purchase.", "I want my money back for this order.",
        "Return the payment for my subscription.", "Request a refund for order ORDER-104.",
        "Can you process a reimbursement for my purchase?", "I would like a refund of ten dollars.",
        "I paid for the wrong plan and want a refund.", "Please return the money I paid yesterday.",
    ],
    "cancel": [
        "Cancel my subscription.", "Stop my plan from renewing.",
        "I want to end my subscription.", "Please turn off automatic renewal.",
        "Close my recurring membership.", "Do not renew my plan next month.",
        "Cancel it.", "Can you cancel this for me?",
    ],
    "account_access": [
        "I cannot log in to my account.", "Help me reset my password.",
        "My login code never arrives.", "I am locked out of my account.",
        "The sign-in page rejects my password.", "How can I recover account access?",
        "My authentication code does not work.", "I forgot my password and cannot sign in.",
    ],
    "unsupported": [
        "What is the weather tomorrow?", "Recommend a stock to buy.",
        "Write a poem about the ocean.", "Give me a pasta recipe.",
        "Book a flight to London.", "Help me solve this algebra problem.",
        "Who won the football match?", "Tell me a joke about cats.",
        "Compare mortgage rates from different banks.", "Teach me to play guitar.",
    ],
}
TRAIN_ROWS = [(text, [label]) for label, texts in TRAIN_GROUPS.items() for text in texts]
TRAIN_ROWS += [
    ("Send my invoice and cancel my subscription.", ["invoice", "cancel"]),
    ("I was charged twice and want the extra payment refunded.", ["duplicate_charge", "refund"]),
    ("Cancel the plan and return my money.", ["cancel", "refund"]),
    ("Send the invoice and help with my declined card.", ["invoice", "payment_failure"]),
    ("Refund my order and recommend a stock.", ["refund", "unsupported"]),
    ("Email my receipt and write a poem.", ["invoice", "unsupported"]),
    ("I cannot log in and I need my latest invoice.", ["account_access", "invoice"]),
    ("My card failed. I also need the last receipt.", ["payment_failure", "invoice"]),
    ("Payment failed and my subscription expires tonight.", ["payment_failure", "urgent"]),
    ("Customers cannot pay; checkout is down for everyone.", ["payment_failure", "urgent"]),
    ("I need the invoice before today's expense deadline.", ["invoice", "urgent"]),
    ("Someone is using my account without permission.", ["account_access", "urgent"]),
    ("My account has been hacked. Please help immediately.", ["account_access", "urgent"]),
    ("I must cancel before renewal in one hour.", ["cancel", "urgent"]),
    ("My payment failed and service will stop today.", ["payment_failure", "urgent"]),
    ("I need a refund today to correct an urgent payment mistake.", ["refund", "urgent"]),
    ("Your billing is terrible. This is my third request for an invoice.", ["invoice", "frustrated"]),
    ("I have told you three times to cancel my subscription.", ["cancel", "frustrated"]),
    ("I am fed up with the repeated payment failures.", ["payment_failure", "frustrated"]),
    ("You charged me twice again. This is unacceptable.", ["duplicate_charge", "frustrated"]),
    ("I am tired of waiting for my refund.", ["refund", "frustrated"]),
    ("I still cannot log in after contacting support repeatedly.", ["account_access", "frustrated"]),
    ("Stop ignoring me and send my receipt.", ["invoice", "frustrated"]),
    ("This is ridiculous. Return my money.", ["refund", "frustrated"]),
    ("Payment is failing again and my subscription ends tonight. I am fed up.",
     ["payment_failure", "urgent", "frustrated"]),
    ("My account is hacked and nobody is helping. This is unacceptable.",
     ["account_access", "urgent", "frustrated"]),
    ("Do not refund the purchase; cancel my subscription instead.", ["cancel"]),
    ("There is no payment failure. I need an invoice.", ["invoice"]),
    ("I am not angry. Please send a receipt when convenient.", ["invoice"]),
    ("No rush. Please cancel my subscription next month.", ["cancel"]),
]
VALIDATION_ROWS = [
    ("May I have my purchase receipt?", ["invoice"]),
    ("There are two identical debits for one purchase.", ["duplicate_charge"]),
    ("The card payment did not complete.", ["payment_failure"]),
    ("Please reimburse this purchase.", ["refund"]),
    ("End my recurring plan.", ["cancel"]),
    ("I need to regain access to my login.", ["account_access"]),
    ("Give me advice on buying shares.", ["unsupported"]),
    ("Send my receipt and stop renewing my subscription.", ["invoice", "cancel"]),
    ("I want my payment returned and a recipe for soup.", ["refund", "unsupported"]),
    ("My renewal payment failed and access ends today.", ["payment_failure", "urgent"]),
    ("An intruder has taken over my account.", ["account_access", "urgent"]),
    ("This is my fourth attempt to get an invoice. I am fed up.", ["invoice", "frustrated"]),
    ("I am sick of this. Your payment screen keeps failing.", ["payment_failure", "frustrated"]),
    ("My refund can wait; I am not in a hurry.", ["refund"]),
]
TEST_ROWS = [
    ("Could you provide a receipt for the annual plan?", ["invoice"]),
    ("One order has appeared as two debits on my card.", ["duplicate_charge"]),
    ("Your checkout will not accept my payment.", ["payment_failure"]),
    ("Please give me back the ten dollars I paid.", ["refund"]),
    ("I would like to discontinue my subscription.", ["cancel"]),
    ("I am unable to get into my account.", ["account_access"]),
    ("Suggest a good hiking trail.", ["unsupported"]),
    ("Give me my receipt and terminate my subscription.", ["invoice", "cancel"]),
    ("Reverse my purchase payment and tell me tomorrow's weather.", ["refund", "unsupported"]),
    ("Checkout failed and my service expires in an hour.", ["payment_failure", "urgent"]),
    ("Someone has broken into my account.", ["account_access", "urgent"]),
    ("I have asked four times for my receipt. This is unacceptable.", ["invoice", "frustrated"]),
    ("Another declined payment! I am fed up with this.", ["payment_failure", "frustrated"]),
    ("I do not want my money back. Just send the invoice.", ["invoice"]),
    ("Please cancel my plan whenever you have time. No urgency.", ["cancel"]),
    ("My card has two charges. Return the extra payment.", ["duplicate_charge", "refund"]),
    ("Cancel my subscription and give me a refund.", ["cancel", "refund"]),
    ("Help with the rejected payment and send the receipt.", ["payment_failure", "invoice"]),
]

def label_matrix(rows):
    return np.asarray([[int(label in labels) for label in LABELS] for _, labels in rows])

for rows in (TRAIN_ROWS, VALIDATION_ROWS, TEST_ROWS):
    assert all(set(labels) <= set(LABELS) for _, labels in rows)
all_texts = [text.casefold().strip() for rows in (TRAIN_ROWS, VALIDATION_ROWS, TEST_ROWS) for text, _ in rows]
assert len(all_texts) == len(set(all_texts)), "Exact text overlap across data splits."
print({"train": len(TRAIN_ROWS), "validation": len(VALIDATION_ROWS), "test": len(TEST_ROWS)})

### 1.3 Train SetFit and select thresholds
A single encoder supplies features for nine binary labels. Intent labels are independent of urgency and frustration labels. SetFit fine-tunes the encoder before fitting the multilabel head. This is real training, not a lookup table or a keyword classifier.

The fixed budget keeps the demo manageable. Do not increase epochs or change thresholds after inspecting the held-out test without creating a new test set.

In [ ]:
import os
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import torch
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
from sklearn.metrics import f1_score
from time import perf_counter

torch.set_num_threads(2)
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
SEED = 42

def train_classifier():
    model = SetFitModel.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, multi_target_strategy="one-vs-rest",
        head_params={"class_weight": "balanced", "max_iter": 1000, "random_state": SEED},
    )
    model.model_body.to("cpu")
    dataset = Dataset.from_dict({"text": [text for text, _ in TRAIN_ROWS],
                                 "label": label_matrix(TRAIN_ROWS).tolist()})
    args = TrainingArguments(
        output_dir="intent_training_output", batch_size=16, num_epochs=1,
        max_steps=40, seed=SEED, report_to="none", save_strategy="no",
        logging_steps=40, use_amp=False,
    )
    trainer = Trainer(model=model, args=args, train_dataset=dataset)
    trainer.train()
    return model

started = perf_counter()
model = train_classifier()
print({"training_seconds": round(perf_counter() - started, 2)})
validation_scores = np.asarray(model.predict_proba([text for text, _ in VALIDATION_ROWS], as_numpy=True))
validation_labels = label_matrix(VALIDATION_ROWS)
thresholds = {}
for index, label in enumerate(LABELS):
    candidates = np.arange(0.25, 0.76, 0.05)
    quality = [f1_score(validation_labels[:, index], validation_scores[:, index] >= t, zero_division=0)
               for t in candidates]
    best = max(range(len(candidates)), key=lambda i: (quality[i], candidates[i]))
    thresholds[label] = round(float(candidates[best]), 2)
print("Validation-selected thresholds:", thresholds)

In [ ]:
effective_thresholds = {**thresholds, **THRESHOLD_OVERRIDES}
config = replace(base_config, thresholds=effective_thresholds)
router = SupportRouter(SetFitScorer(model), config)
print({"policy_id": config.policy_id, "effective_thresholds": effective_thresholds})

### 1.4 Inspect the independent decisions
Classification does not authorize a payment, cancellation, or refund. `route` means the request can go to a handler for further checks. `handoff` is the safe fallback for classifier failure or critical review; ordinary uncertainty gets bounded clarification.

Inputs longer than the character or encoder token limit are not silently truncated and routed. This example handles English text and does not implement automatic language identification.

In [ ]:
examples = [
    ("Cancel it.", ConversationContext()),
    ("Cancel it.", ConversationContext(target="subscription")),
    ("Send my invoice and cancel my subscription.", ConversationContext()),
    ("Please refund my purchase and recommend a stock.", ConversationContext(order_reference="ORDER-104")),
    ("My payment failed and access ends tonight.", ConversationContext()),
]
for number, (text, context) in enumerate(examples, 1):
    print(number, json.dumps(router.classify(text, context).audit()))

# Part 2 — End-to-end support application

The application accepts a message, classifies it, asks for missing context, and routes a complete request to mock support queues. A slot-only answer resumes all pending business intents rather than losing the second task.

Queue handlers are intentionally mock handlers. A production refund handler must independently authenticate the user, check order ownership, enforce refund limits, and use idempotency. No classifier score substitutes for those checks.

The application does not implement a second PII detector. Feed it text released by the PII guard from notebook 01. Retain only the minimum approved structured state needed for routing; an order reference may itself require protected storage.

In [ ]:
class SupportApplication:
    def __init__(self, router):
        self.router = router
        self.context = ConversationContext()
        self.handler_calls = []

    def handle(self, message):
        decision = self.router.classify(message, self.context)
        if decision.action == "clarify":
            fields = extract_fields(message, self.context)
            self.context = ConversationContext(
                target=fields["target"], order_reference=fields["order_reference"],
                pending_intents=decision.intents, pending_unsupported=decision.scope == "partial",
                clarification_count=self.context.clarification_count + 1,
            )
            if decision.clarity == "conflicting":
                answer = "Do you want to stop renewal or keep the subscription renewing?"
            elif "target" in decision.missing_fields:
                answer = "Do you want to cancel a subscription or an order?"
            elif "order_reference" in decision.missing_fields:
                answer = "What is the order reference? Please do not send card or bank details."
            else:
                answer = "Is this about an invoice, payment, refund, subscription, or account access?"
        elif decision.action in ("route", "route_supported"):
            self.handler_calls.extend(decision.routes)
            answer = "Request queued for: " + ", ".join(decision.routes) + "."
            if decision.scope == "partial":
                answer += " The unrelated request is outside this assistant's scope."
            self.context = ConversationContext()
        elif decision.action == "redirect":
            answer = "I can help with this product's billing, payments, subscriptions, and account access."
            self.context = ConversationContext()
        else:
            answer = "I will route this conversation for human review."
            self.context = ConversationContext()
        return {"decision": decision.audit(), "answer": answer}


app = SupportApplication(router)
for message in ("Please refund my purchase.", "ORDER-104"):
    response = app.handle(message)
    print(json.dumps(response, indent=2))
print("Mock handler calls:", app.handler_calls)

### 2.1 Verify deterministic behavior
These tests inject known classifier outputs so failures point to routing logic rather than model quality. They test clarification, mixed scope, multiple intents, conflicts, urgency, frustration, unavailable classifiers, invalid input, privacy-safe audit fields, and conversation state.

**Expected: 16 policy tests pass.** The semantic and routing metrics below use the real trained model. They can show failures even when all policy tests pass.

In [ ]:
import unittest


class FixedScorer:
    def __init__(self, labels=(), fail=False):
        self.labels, self.fail = labels, fail

    def scores(self, text):
        if self.fail:
            raise RuntimeError("Classifier unavailable")
        return {label: 0.99 if label in self.labels else 0.01 for label in LABELS}


class PolicyTests(unittest.TestCase):
    def route(self, text, labels=(), context=None, **config):
        return SupportRouter(FixedScorer(labels), RouterConfig(**config)).classify(text, context)

    def test_uncertainty_is_not_rejection(self):
        r = self.route("Could you help with this?")
        self.assertEqual((r.scope, r.action), ("uncertain", "clarify"))

    def test_cancel_needs_target_then_resolves(self):
        r = self.route("Cancel it.", ["cancel"])
        self.assertEqual((r.clarity, r.action), ("missing_context", "clarify"))
        r = self.route("Cancel it.", ["cancel"], ConversationContext(target="subscription"))
        self.assertEqual(r.action, "route")

    def test_mixed_request_is_not_confusion(self):
        r = self.route("Send my invoice and cancel my subscription.", ["invoice", "cancel"])
        self.assertTrue(r.multiple_requests)
        self.assertEqual((r.clarity, r.action), ("clear", "route"))
        self.assertEqual(r.routes, ("billing", "subscriptions"))

    def test_partial_scope_preserves_supported_intent(self):
        r = self.route("Send my invoice and suggest a stock.", ["invoice", "unsupported"])
        self.assertEqual((r.scope, r.action), ("partial", "route_supported"))

    def test_scope_is_configurable(self):
        r = self.route("Cancel my subscription.", ["cancel"], supported_intents=("invoice",))
        self.assertEqual((r.scope, r.action), ("unsupported", "redirect"))

    def test_urgent_is_not_authorization(self):
        r = self.route("Refund me immediately.", ["refund", "urgent"])
        self.assertEqual((r.urgency, r.action), ("time_sensitive", "clarify"))
        self.assertIn("order_reference", r.missing_fields)

    def test_frustration_is_not_out_of_scope(self):
        r = self.route("Send my invoice. I am fed up.", ["invoice", "frustrated"])
        self.assertEqual((r.scope, r.action), ("supported", "handoff"))

    def test_critical_review_precedes_missing_fields(self):
        r = self.route("My account is hacked. Refund the purchase.", ["account_access", "refund"])
        self.assertEqual((r.action, r.routes), ("handoff", ("incident_review",)))

    def test_clarification_limit(self):
        context = ConversationContext(clarification_count=2)
        self.assertEqual(self.route("Help with this.", context=context).action, "handoff")

    def test_conflicting_request(self):
        r = self.route("Cancel my subscription but keep renewing it.", ["cancel"])
        self.assertEqual((r.clarity, r.action), ("conflicting", "clarify"))

    def test_failure_never_reaches_handler(self):
        app = SupportApplication(SupportRouter(FixedScorer(fail=True), RouterConfig()))
        self.assertEqual(app.handle("Refund me.")["decision"]["action"], "handoff")
        self.assertEqual(app.handler_calls, [])

    def test_invalid_input(self):
        for text in (None, "", "x" * 2001):
            self.assertEqual(self.route(text).reason, "INVALID_OR_OVERSIZED_INPUT")
        self.assertEqual(self.route("Help me", context="").reason, "INVALID_CONTEXT")

    def test_reference_reply_preserves_multiple_intents(self):
        context = ConversationContext(target="subscription", pending_intents=("refund", "cancel"))
        r = self.route("ORDER-104", context=context)
        self.assertEqual(set(r.intents), {"refund", "cancel"})
        self.assertEqual(r.action, "route")

    def test_application_tracks_clarification(self):
        app = SupportApplication(SupportRouter(FixedScorer(["cancel"]), RouterConfig()))
        self.assertEqual(app.handle("Cancel it.")["decision"]["action"], "clarify")
        self.assertEqual(app.context.clarification_count, 1)
        self.assertEqual(app.handle("subscription")["decision"]["action"], "route")
        self.assertEqual(app.handler_calls, ["subscriptions"])

    def test_audit_excludes_message_and_reference(self):
        r = self.route("Refund ORDER-104 please.", ["refund"])
        payload = json.dumps(r.audit())
        self.assertNotIn("ORDER-104", payload)
        self.assertNotIn("Refund ORDER", payload)

    def test_invalid_configuration(self):
        for values in ({"max_clarifications": 0}, {"thresholds": {"refund": float("nan")}},
                       {"supported_intents": ("unknown",)}):
            with self.subTest(values=values), self.assertRaises(ValueError):
                RouterConfig(**values)


verification = unittest.TextTestRunner(verbosity=1).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(PolicyTests))
assert verification.wasSuccessful(), "Policy checks failed."
print(f"Policy checks passed: {verification.testsRun}")

### 2.2 Measure classifier and application performance

| Measurement | Meaning |
|---|---|
| Per-label precision/recall/F1 | Error rates for each intent, urgency, frustration, and unsupported examples |
| Intent exact match | Every business-intent label agrees for a message |
| False out-of-scope rejection | Supported or partial cases labeled wholly unsupported |
| Unsupported acceptance | Wholly unsupported cases sent to a support handler |
| Unnecessary clarification | Cases ready to route that instead receive a clarification |
| Urgent miss / false urgency | Missed urgent cases and nonurgent cases incorrectly prioritized |
| Scope/action accuracy | Agreement with explicit scenario expectations |
| Uncertain / handoff / error rates | Abstention, human review, and failed classification, reported separately |
| Warm p50/p95 latency | Sequential end-to-end classification time, excluding training and model load |

These small synthetic evaluations demonstrate how to measure performance; they are not a benchmark claim. Routing scenarios reuse some semantic test messages, so the two reports are not independent datasets. Mismatch IDs are printed without hiding failures. No generative fallback is configured; handoff is measured instead of an LLM fallback rate.

In [ ]:
from sklearn.metrics import classification_report, precision_recall_fscore_support, accuracy_score
from statistics import median
from math import ceil


def fraction(numerator, denominator):
    return round(numerator / denominator, 4) if denominator else None


def evaluate_model(model, rows, thresholds):
    expected = label_matrix(rows)
    scores = np.asarray(model.predict_proba([text for text, _ in rows], as_numpy=True))
    predicted = scores >= np.asarray([thresholds.get(label, 0.5) for label in LABELS])
    print(classification_report(expected, predicted, target_names=LABELS, zero_division=0, digits=3))
    metrics = {}
    for name, indices in (("business_intents", range(len(INTENTS))), ("all_labels", range(len(LABELS)))):
        p, r, f, _ = precision_recall_fscore_support(expected[:, indices], predicted[:, indices],
                                                   average="micro", zero_division=0)
        metrics[name] = {"micro_precision": round(float(p), 4), "micro_recall": round(float(r), 4),
                         "micro_f1": round(float(f), 4),
                         "exact_match": round(float(accuracy_score(expected[:, indices], predicted[:, indices])), 4)}
    metrics["mismatched_example_ids"] = np.flatnonzero(np.any(expected != predicted, axis=1)).tolist()
    return metrics


print("Held-out semantic evaluation:")
semantic_metrics = evaluate_model(model, TEST_ROWS, effective_thresholds)
print(json.dumps(semantic_metrics, indent=2))

ROUTING_CASES = [
    {"id": "invoice", "text": "Could you provide a receipt for the annual plan?", "scope": "supported", "action": "route", "urgent": False},
    {"id": "missing_target", "text": "Cancel it.", "scope": "supported", "action": "clarify", "urgent": False},
    {"id": "known_target", "text": "Cancel it.", "context": ConversationContext(target="subscription"), "scope": "supported", "action": "route", "urgent": False},
    {"id": "two_requests", "text": "Give me my receipt and terminate my subscription.", "scope": "supported", "action": "route", "urgent": False},
    {"id": "mixed_scope", "text": "Reverse my purchase payment and tell me tomorrow's weather.", "context": ConversationContext(order_reference="ORDER-104"), "scope": "partial", "action": "route_supported", "urgent": False},
    {"id": "unsupported", "text": "Suggest a good hiking trail.", "scope": "unsupported", "action": "redirect", "urgent": False},
    {"id": "time_sensitive", "text": "Checkout failed and my service expires in an hour.", "scope": "supported", "action": "route", "urgent": True},
    {"id": "critical", "text": "My account is hacked.", "scope": "supported", "action": "handoff", "urgent": True},
    {"id": "frustrated", "text": "I have asked four times for my receipt. This is unacceptable.", "scope": "supported", "action": "handoff", "urgent": False},
    {"id": "vague", "text": "Can you help with that thing?", "scope": "uncertain", "action": "clarify", "urgent": False},
    {"id": "clarification_limit", "text": "Cancel it.", "context": ConversationContext(clarification_count=2), "scope": "supported", "action": "handoff", "urgent": False},
    {"id": "refund_missing_ref", "text": "Please give me back the ten dollars I paid.", "scope": "supported", "action": "clarify", "urgent": False},
    {"id": "refund_with_ref", "text": "Please give me back the ten dollars I paid.", "context": ConversationContext(order_reference="ORDER-104"), "scope": "supported", "action": "route", "urgent": False},
]


def evaluate_router(router, cases):
    records = [(case, router.classify(case["text"], case.get("context"))) for case in cases]
    supported = [(c, r) for c, r in records if c["scope"] in ("supported", "partial")]
    urgent = [(c, r) for c, r in records if c["urgent"]]
    complete = [(c, r) for c, r in records if c["action"] in ("route", "route_supported")]
    mismatches = [{"id": c["id"], "expected_action": c["action"], "actual_action": r.action,
                   "expected_scope": c["scope"], "actual_scope": r.scope}
                  for c, r in records if c["action"] != r.action or c["scope"] != r.scope]
    return {
        "examples": len(records), "supported_examples": len(supported), "urgent_examples": len(urgent),
        "unsupported_acceptance_rate": fraction(sum(c["scope"] == "unsupported" and r.action in ("route", "route_supported") for c, r in records), sum(c["scope"] == "unsupported" for c, _ in records)),
        "false_urgency_rate": fraction(sum(not c["urgent"] and r.urgency != "routine" for c, r in records), sum(not c["urgent"] for c, _ in records)),
        "classifier_error_rate": fraction(sum(r.reason == "CLASSIFIER_UNAVAILABLE" for _, r in records), len(records)),
        "scope_accuracy": fraction(sum(c["scope"] == r.scope for c, r in records), len(records)),
        "action_accuracy": fraction(sum(c["action"] == r.action for c, r in records), len(records)),
        "false_out_of_scope_rejection_rate": fraction(sum(r.scope == "unsupported" for _, r in supported), len(supported)),
        "unnecessary_clarification_rate": fraction(sum(r.action == "clarify" for _, r in complete), len(complete)),
        "urgent_miss_rate": fraction(sum(r.urgency == "routine" for _, r in urgent), len(urgent)),
        "uncertain_scope_rate": fraction(sum(r.scope == "uncertain" for _, r in records), len(records)),
        "human_handoff_rate": fraction(sum(r.action == "handoff" for _, r in records), len(records)),
        "mismatches": mismatches,
    }


routing_metrics = evaluate_router(router, ROUTING_CASES)
print("Held-out routing scenarios:", json.dumps(routing_metrics, indent=2))
router.classify("Please send my invoice.")
timings = []
for _ in range(3):
    for case in ROUTING_CASES:
        timings.append(router.classify(case["text"], case.get("context")).latency_ms)
print({"warm_scans": len(timings), "median_ms": round(median(timings), 2),
       "p95_ms": round(sorted(timings)[ceil(len(timings) * 0.95) - 1], 2)})

**Read the failures before the aggregate score.** In the recorded run, business-intent micro F1 was 0.872, but both held-out duplicate-charge examples were missed. The classifier also missed the unsupported portion of one mixed-scope request. Routing action accuracy was 12/13; that aggregate hides the mixed-scope failure. These are results from a small synthetic set, not deployment performance. Expand the duplicate-charge and mixed-request training/validation data, then evaluate on a new held-out set rather than tuning against these test examples.

### 2.3 Add a sample and retain the configuration
Edit the synthetic sample below. Use `ConversationContext` for a target or reference already collected by your application; do not paste full conversation history into the encoder. For long or multi-turn requests beyond this example's limits, add explicit context resolution and evaluate it separately.

In [ ]:
CUSTOM_TEXT = "Cancel my subscription and send my invoice."
CUSTOM_CONTEXT = ConversationContext()
print(json.dumps(router.classify(CUSTOM_TEXT, CUSTOM_CONTEXT).audit(), indent=2))

from importlib.metadata import version
manifest = {
    "profile_version": "1.0.0",
    "policy_id": config.policy_id,
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "seed": SEED,
    "train_examples": len(TRAIN_ROWS),
    "training_data_sha256": hashlib.sha256(json.dumps(TRAIN_ROWS).encode()).hexdigest(),
    "packages": {name: version(name) for name in
                 ("setfit", "sentence-transformers", "transformers", "torch", "scikit-learn")},
    "policy": config.document(),
}
print(json.dumps(manifest, indent=2))

### Before connecting real handlers
- Tune thresholds on validation data and keep a separate, representative test set. Report intent and urgency/frustration performance separately. Unknown-topic coverage remains open-ended; low error on these examples does not establish general out-of-scope detection.
- Validate the structured context at the application boundary. The example tracks one conversation per `SupportApplication` instance. Use isolated, expiring session state in a service; never share one instance across users.
- The conflict detector, field extraction, and slot-reply recognition intentionally cover narrow patterns. Paraphrases, negation, and compound follow-ups need additional labeled cases or a bounded clarification/resolution component.
- Run CPU inference in bounded workers when serving async requests. Apply deadlines and queue limits; this notebook has no hard inference timeout. A circuit breaker should lead to review or a defined fallback, not unchecked execution.
- Never let urgency or frustration bypass ownership checks or increase transaction permissions. Frustration is a model signal, not a fact about the customer; configure whether it changes tone, priority, or handoff.
- Audit decisions and aggregate outcomes. Do not log message text, scores with source snippets, references, or full conversation context by default. This notebook's synthetic outputs can be retained; review notebook and framework tracing before using real records.
- `policy_id` identifies policy configuration only. Retain the model weights, dataset version, manifest, dependency lockfile, and code revision together for reproducible deployment. Retraining can vary across hardware and library versions even with a fixed seed.

**References:** [SetFit multilabel classification](https://huggingface.co/docs/setfit/how_to/multilabel) · [Classification heads](https://huggingface.co/docs/setfit/how_to/classification_heads) · [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)